# ARIA · `aria-personal-manager-v1` — TRL GRPO training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Indrajeety993648/Aria/blob/main/backend/training/aria_train_colab.ipynb)

**Runtime:** Colab T4 (free) or A100. Set `Runtime → Change runtime type → T4 GPU` first.

This notebook fine-tunes **Qwen 2.5 0.5B-Instruct** with **TRL GRPO** + LoRA on the ARIA RL environment, then plots the reward curve and runs an ablation showing the `relationship_health` dimension is what teaches the agent the relationship-aware behaviour.

**The story we want to tell:**

* Most LLM-agent benchmarks reward task completion only.
* ARIA penalises completing tasks at the cost of damaging relationships.
* Two training runs — same setup, only difference is whether the relationship dimension is zeroed.
* The reward curves on the same axes are the proof.

## 0 · GPU sanity check

Run this first. If it complains, switch the runtime to GPU and re-run.

In [1]:
!nvidia-smi || echo 'NO GPU — switch the runtime'

Sat Apr 25 09:37:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1 · Install dependencies and clone the repo

Pinned versions to keep tomorrow's run reproducible. ~2 minutes.

In [2]:
%%bash
set -eux
pip install -q --upgrade \
    'transformers>=4.45,<4.50' \
    'trl>=0.13,<0.15' \
    'peft>=0.12,<0.14' \
    'accelerate>=1.0,<2.0' \
    'bitsandbytes>=0.43' \
    'datasets>=3.0' \
    'openenv-core>=0.1' \
    matplotlib wandb

if [ ! -d /content/Aria ]; then
    git clone --depth 1 https://github.com/Indrajeety993648/Aria.git /content/Aria
fi

cd /content/Aria
pip install -q -e backend/packages/aria-contracts
pip install -q -e backend/packages/aria-scenarios
pip install -q -e backend/packages/aria-rewards
pip install -q -e backend/services/env-service
echo OK

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.9/313.9 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.6/174.6 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 127.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.2/27.2 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

+ pip install -q --upgrade 'transformers>=4.45,<4.50' 'trl>=0.13,<0.15' 'peft>=0.12,<0.14' 'accelerate>=1.0,<2.0' 'bitsandbytes>=0.43' 'datasets>=3.0' 'openenv-core>=0.1' matplotlib wandb
+ '[' '!' -d /content/Aria ']'
+ git clone --depth 1 https://github.com/Indrajeety993648/Aria.git /content/Aria
Cloning into '/content/Aria'...
+ cd /content/Aria
+ pip install -q -e backend/packages/aria-contracts
+ pip install -q -e backend/packages/aria-scenarios
+ pip install -q -e backend/packages/aria-rewards
+ pip install -q -e backend/services/env-service
+ echo OK


## 2 · Smoke test the env

If this prints six rubrics, the env is wired correctly.

In [3]:
import sys, os, importlib

os.chdir('/content/Aria')
for p in [
    '/content/Aria/backend',
    '/content/Aria/backend/packages/aria-contracts/src',
    '/content/Aria/backend/packages/aria-scenarios/src',
    '/content/Aria/backend/packages/aria-rewards/src',
    '/content/Aria/backend/services/env-service/src',
]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Drop any half-loaded modules from a previous failed import.
for mod in list(sys.modules):
    if mod.startswith(('aria_contracts', 'aria_scenarios', 'aria_rewards', 'env_service')):
        del sys.modules[mod]

from env_service.aria_env import AriaEnv
from aria_contracts import AriaAction, ActionId

env = AriaEnv()
obs = env.reset(seed=42, category='calendar_conflict', difficulty='medium')
print('rubric tree:')
for name, r in env.rubric.named_rubrics():
    print(f'  {name:<22} weight={r.weight}')

out = env.step(AriaAction(action_id=ActionId.RESOLVE_CONFLICT.value, target_id='conflict_personal'))
print(f'\nstep reward = {out.reward:+.4f}')
print(f'breakdown   = {out.reward_breakdown}')

rubric tree:
  task_completion        weight=0.25
  relationship_health    weight=0.2
  user_satisfaction      weight=0.2
  time_efficiency        weight=0.15
  conflict_resolution    weight=0.15
  safety                 weight=0.05

step reward = +0.6000
breakdown   = task_completion=0.3999999999999999 relationship_health=-0.050000000000000044 user_satisfaction=1.8 time_efficiency=0.0 conflict_resolution=1.0 safety=0.0 total=0.6


## 3 · (Optional) wandb login

Skip if you don't have a wandb account — local CSV logging is automatic.

In [ ]:
import os
# Uncomment + paste your wandb key:
# os.environ['WANDB_API_KEY'] = 'xxx'
# !wandb login --relogin
print('wandb key set' if os.getenv('WANDB_API_KEY') else 'wandb skipped (local CSV only)')

## 4 · Run A — full reward (~6h on T4 for 500 steps)

**Tip:** for a quick smoke run first, lower `--steps` to 30. That should land in ~15 min on T4 and produce a non-flat curve.

In [4]:
%%bash
cd /content/Aria
mkdir -p checkpoints logs
python -m backend.training.train_grpo \
    --run-name aria-full \
    --steps 500 \
    --output-dir ./checkpoints/aria-full \
    2>&1 | tee logs/run-a.log

2026-04-25 09:39:15.756322: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777109955.777463    8657 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777109955.784770    8657 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777109955.803298    8657 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777109955.803332    8657 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777109955.803336    8657 computation_placer.cc:177] computation placer alr

## 5 · Run B — ablation (`relationship_health` zeroed)

Same hyperparameters; only `--ablate relationship_health` differs. The dimension still computes (so you can read its `last_score` for analysis), but contributes 0 to the surfaced reward.

In [ ]:
%%bash
cd /content/Aria
python -m backend.training.train_grpo \
    --run-name aria-ablate-rh \
    --ablate relationship_health \
    --steps 500 \
    --output-dir ./checkpoints/aria-ablate-rh \
    2>&1 | tee logs/run-b.log

## 6 · Plot the two curves on the same axes

**The money plot.** Full-reward agent should plateau higher; ablated agent reward-hacks via unilateral cancels.

In [ ]:
%%bash
cd /content/Aria
python backend/training/plot.py \
    --full-dir ./checkpoints/aria-full \
    --ablate-dir ./checkpoints/aria-ablate-rh \
    --baselines ./backend/baselines/baseline_metrics.json \
    --out-dir ./docs/assets

In [ ]:
from IPython.display import Image, display
display(Image('/content/Aria/docs/assets/reward_curve.png'))

## 7 · Evaluate the trained checkpoints

Runs both checkpoints across all 18 (category, difficulty) cells. Outputs `eval_summary.json` per run + a per-category bar chart.

In [ ]:
%%bash
cd /content/Aria
python -m backend.training.eval \
    --model-path ./checkpoints/aria-full/final \
    --out-dir ./eval/full
python -m backend.training.eval \
    --model-path ./checkpoints/aria-ablate-rh/final \
    --ablate relationship_health \
    --out-dir ./eval/ablate
python backend/training/plot.py \
    --full-dir ./checkpoints/aria-full \
    --ablate-dir ./checkpoints/aria-ablate-rh \
    --eval-full ./eval/full/eval_summary.json \
    --eval-ablate ./eval/ablate/eval_summary.json \
    --baselines ./backend/baselines/baseline_metrics.json \
    --out-dir ./docs/assets

In [ ]:
from IPython.display import Image, display
display(Image('/content/Aria/docs/assets/category_winrate.png'))

## 8 · Inspect a qualitative trajectory

Pick the canonical conflict scenario and dump the trained agent's first 8 actions side by side.

In [ ]:
import json
from pathlib import Path

for label, path in [
    ('FULL  ', './eval/full/trajectories/calendar_conflict_medium_seed00.json'),
    ('ABLATE', './eval/ablate/trajectories/calendar_conflict_medium_seed00.json'),
]:
    if Path(path).exists():
        traj = json.loads(Path(path).read_text())
        print(f'\n[{label}] first 8 steps')
        for step in traj[:8]:
            print(f"  action_id={step['action_id']:<2} target={step['target_id']:<22} reward={step['reward']:+.3f}")

## What you've shown

1. The env is OpenEnv-compliant (six inspectable rubrics).
2. TRL GRPO can train an LLM agent on it end-to-end.
3. Removing the `relationship_health` dimension changes what the agent learns — the failure mode is recognizable: more `CANCEL`, fewer `PROPOSE_ALTERNATIVE`.

Continue:

* Repo: https://github.com/Indrajeety993648/Aria
* HF Space: https://huggingface.co/spaces/indra123/aria-personal-manager-v1
* HF Blog: https://huggingface.co/blog/indra123/aria